In [16]:
import os
import pandas as pd
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
from dotenv import load_dotenv

In [17]:
load_dotenv()

# --- 1. SET UP YOUR SNOWFLAKE CREDENTIALS ---
conn = snowflake.connector.connect(
    user=os.getenv('SNOWFLAKE_USER'),
    password=os.getenv('SNOWFLAKE_PASSWORD'),
    account=os.getenv('SNOWFLAKE_ACCOUNT'),
    warehouse=os.getenv('SNOWFLAKE_WAREHOUSE'),
    database=os.getenv('SNOWFLAKE_DATABASE'),
    schema=os.getenv('SNOWFLAKE_SCHEMA'),
    role=os.getenv('SNOWFLAKE_ROLE')
)

## 1. Impossible / Suspicious Dates

Check for dates that violate logical order or business rules.

In [18]:
# 1.1 Orders: Impossible date sequences
query = """
SELECT 
    'delivered_before_ordered' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE delivered_date < order_date

UNION ALL

SELECT 
    'approved_before_ordered' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE approval_date < order_date

UNION ALL

SELECT 
    'shipped_before_approved' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE shipped_date < approval_date

UNION ALL

SELECT 
    'delivered_before_shipped' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE delivered_date < shipped_date

UNION ALL

SELECT 
    'estimated_delivery_before_ordered' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE estimated_delivery_date < order_date;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2072699288.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,delivered_before_ordered,130
1,approved_before_ordered,0
2,shipped_before_approved,0
3,delivered_before_shipped,130
4,estimated_delivery_before_ordered,0


In [19]:
# 1.2 Returns: Date sequence checks
query = """
SELECT 
    'return_received_before_initiated' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."07_RETURNS_CANCELLATIONS"
WHERE return_received_date < return_initiated_date;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\630067854.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,return_received_before_initiated,0


In [20]:
# 1.3 Support Tickets: resolved before created
query = """
SELECT 
    'resolved_before_created' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE resolved_date < created_date;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\1331036897.py:10: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,resolved_before_created,0


## 2. Invalid Numeric Values

Check for negative, zero, or out-of-range values where they shouldn't exist.

In [21]:
# 2.1 Order Items: Invalid prices and quantities
query = """
SELECT 
    'negative_or_zero_unit_price' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE unit_price <= 0

UNION ALL

SELECT 
    'negative_or_zero_item_total' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE item_total <= 0

UNION ALL

SELECT 
    'negative_or_zero_quantity' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE quantity <= 0

UNION ALL

SELECT 
    'negative_item_weight' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE item_weight_kg < 0

UNION ALL

SELECT 
    'negative_discount_pct' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE discount_pct < 0

UNION ALL

SELECT 
    'discount_over_100_pct' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."02_ORDER_ITEMS"
WHERE discount_pct > 100;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\729962239.py:50: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_or_zero_unit_price,0
1,negative_or_zero_item_total,0
2,negative_or_zero_quantity,0
3,negative_item_weight,0
4,negative_discount_pct,0
5,discount_over_100_pct,0


In [22]:
# 2.2 Orders: Invalid totals and costs
query = """
SELECT 
    'negative_or_zero_order_total' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE order_total <= 0

UNION ALL

SELECT 
    'negative_shipping_cost' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE shipping_cost < 0

UNION ALL

SELECT 
    'negative_total_weight' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE total_weight_kg < 0

UNION ALL

SELECT 
    'negative_or_zero_num_items' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE num_items <= 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2434732340.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_or_zero_order_total,0
1,negative_shipping_cost,0
2,negative_total_weight,0
3,negative_or_zero_num_items,0


In [23]:
# 2.3 Products: Invalid prices and dimensions
query = """
SELECT 
    'negative_or_zero_price' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."03_PRODUCTS"
WHERE price <= 0

UNION ALL

SELECT 
    'negative_or_zero_cost_price' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."03_PRODUCTS"
WHERE cost_price <= 0

UNION ALL

SELECT 
    'cost_exceeds_price' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."03_PRODUCTS"
WHERE cost_price > price

UNION ALL

SELECT 
    'negative_weight' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."03_PRODUCTS"
WHERE weight_kg < 0

UNION ALL

SELECT 
    'negative_dimensions' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."03_PRODUCTS"
WHERE length_cm < 0 OR width_cm < 0 OR height_cm < 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\704620113.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_or_zero_price,0
1,negative_or_zero_cost_price,0
2,cost_exceeds_price,0
3,negative_weight,0
4,negative_dimensions,0


In [24]:
# 2.4 Customer Reviews: Rating out of range (should be 1-5)
query = """
SELECT 
    'rating_out_of_range' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."08_CUSTOMER_REVIEWS"
WHERE rating < 1 OR rating > 5

UNION ALL

SELECT 
    'delivery_rating_out_of_range' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."08_CUSTOMER_REVIEWS"
WHERE delivery_rating < 1 OR delivery_rating > 5

UNION ALL

SELECT 
    'product_rating_out_of_range' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."08_CUSTOMER_REVIEWS"
WHERE product_rating < 1 OR product_rating > 5

UNION ALL

SELECT 
    'negative_days_late' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."08_CUSTOMER_REVIEWS"
WHERE days_late < 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\3541591363.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,rating_out_of_range,0
1,delivery_rating_out_of_range,0
2,product_rating_out_of_range,0
3,negative_days_late,0


In [25]:
# 2.5 Sellers: Invalid scores and processing hours
query = """
SELECT 
    'quality_score_out_of_range' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."05_SELLERS"
WHERE quality_score < 1 OR quality_score > 5

UNION ALL

SELECT 
    'negative_processing_hours' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."05_SELLERS"
WHERE avg_processing_hours < 0

UNION ALL

SELECT 
    'negative_return_policy_days' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."05_SELLERS"
WHERE return_policy_days < 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\4228151323.py:26: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,quality_score_out_of_range,0
1,negative_processing_hours,0
2,negative_return_policy_days,0


In [26]:
# 2.6 Inventory Snapshots: Invalid quantities
query = """
SELECT 
    'negative_quantity_on_hand' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS"
WHERE quantity_on_hand < 0

UNION ALL

SELECT 
    'negative_quantity_reserved' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS"
WHERE quantity_reserved < 0

UNION ALL

SELECT 
    'negative_quantity_available' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS"
WHERE quantity_available < 0

UNION ALL

SELECT 
    'reserved_exceeds_on_hand' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS"
WHERE quantity_reserved > quantity_on_hand

UNION ALL

SELECT 
    'available_calculation_mismatch' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS"
WHERE quantity_available != (quantity_on_hand - quantity_reserved);
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\3354085724.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_quantity_on_hand,0
1,negative_quantity_reserved,0
2,negative_quantity_available,0
3,reserved_exceeds_on_hand,0
4,available_calculation_mismatch,1992


In [27]:
# 2.7 Returns: Invalid refund amounts
query = """
SELECT 
    'negative_refund_amount' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."07_RETURNS_CANCELLATIONS"
WHERE refund_amount < 0

UNION ALL

SELECT 
    'negative_restocking_fee' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."07_RETURNS_CANCELLATIONS"
WHERE restocking_fee < 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2426889756.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_refund_amount,0
1,negative_restocking_fee,0


In [28]:
# 2.8 Marketing Campaigns: Invalid metrics
query = """
SELECT 
    'negative_spend' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."10_MARKETING_CAMPAIGNS"
WHERE spend < 0

UNION ALL

SELECT 
    'negative_impressions' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."10_MARKETING_CAMPAIGNS"
WHERE impressions < 0

UNION ALL

SELECT 
    'negative_clicks' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."10_MARKETING_CAMPAIGNS"
WHERE clicks < 0

UNION ALL

SELECT 
    'clicks_exceed_impressions' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."10_MARKETING_CAMPAIGNS"
WHERE clicks > impressions

UNION ALL

SELECT 
    'conversions_exceed_clicks' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."10_MARKETING_CAMPAIGNS"
WHERE conversions > clicks;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\4248716514.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_spend,0
1,negative_impressions,0
2,negative_clicks,0
3,clicks_exceed_impressions,0
4,conversions_exceed_clicks,0


In [29]:
# 2.9 Support Tickets: Invalid metrics
query = """
SELECT 
    'negative_resolution_hours' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE resolution_hours < 0

UNION ALL

SELECT 
    'satisfaction_score_out_of_range' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE satisfaction_score < 1 OR satisfaction_score > 5

UNION ALL

SELECT 
    'negative_first_response_minutes' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE first_response_minutes < 0

UNION ALL

SELECT 
    'negative_num_interactions' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE num_interactions < 0;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\385579403.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,negative_resolution_hours,0
1,satisfaction_score_out_of_range,0
2,negative_first_response_minutes,0
3,negative_num_interactions,0


## 3. Status / Date Consistency

Check for mismatches between status fields and their corresponding timestamps.

In [30]:
# 3.1 Orders: Status vs Date consistency
query = """
SELECT 
    'delivered_status_no_delivered_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE order_status = 'Delivered' AND delivered_date IS NULL

UNION ALL

SELECT 
    'shipped_status_no_shipped_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE order_status IN ('Shipped', 'Delivered') AND shipped_date IS NULL

UNION ALL

SELECT 
    'has_delivered_date_but_not_delivered_status' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE delivered_date IS NOT NULL AND order_status != 'Delivered'

UNION ALL

SELECT 
    'cancelled_but_has_delivered_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."01_ORDERS"
WHERE order_status = 'Cancelled' AND delivered_date IS NOT NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\3973008436.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,delivered_status_no_delivered_date,0
1,shipped_status_no_shipped_date,0
2,has_delivered_date_but_not_delivered_status,0
3,cancelled_but_has_delivered_date,0


In [31]:
# 3.2 Support Tickets: Status vs Date consistency
query = """
SELECT 
    'resolved_status_no_resolved_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE status = 'Resolved' AND resolved_date IS NULL

UNION ALL

SELECT 
    'has_resolved_date_but_not_resolved_status' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE resolved_date IS NOT NULL AND status != 'Resolved'

UNION ALL

SELECT 
    'resolved_but_no_resolution_hours' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE status = 'Resolved' AND resolution_hours IS NULL

UNION ALL

SELECT 
    'resolved_but_no_satisfaction_score' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."11_SUPPORT_TICKETS"
WHERE status = 'Resolved' AND satisfaction_score IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\829699971.py:34: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,resolved_status_no_resolved_date,0
1,has_resolved_date_but_not_resolved_status,0
2,resolved_but_no_resolution_hours,0
3,resolved_but_no_satisfaction_score,0


In [32]:
# 3.3 Returns: Status vs Date consistency
query = """
SELECT 
    'completed_status_no_received_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."07_RETURNS_CANCELLATIONS"
WHERE return_status = 'Completed' AND return_received_date IS NULL

UNION ALL

SELECT 
    'in_transit_but_has_received_date' AS check_name,
    COUNT(*) AS violation_count
FROM sales_raw."07_RETURNS_CANCELLATIONS"
WHERE return_status = 'In Transit' AND return_received_date IS NOT NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\1353687865.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATION_COUNT
0,completed_status_no_received_date,289
1,in_transit_but_has_received_date,333


## 4. Duplicate Primary Keys

Check for duplicate values in columns that should be unique identifiers.

In [33]:
# 4.1 Check duplicate primary keys across all tables
query = """
SELECT '01_orders.order_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT order_id FROM sales_raw."01_ORDERS" GROUP BY order_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '02_order_items.order_item_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT order_item_id FROM sales_raw."02_ORDER_ITEMS" GROUP BY order_item_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '03_products.product_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT product_id FROM sales_raw."03_PRODUCTS" GROUP BY product_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '04_customers.customer_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT customer_id FROM sales_raw."04_CUSTOMERS" GROUP BY customer_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '05_sellers.seller_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT seller_id FROM sales_raw."05_SELLERS" GROUP BY seller_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '06_shipping.event_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT event_id FROM sales_raw."06_SHIPPING_LOGISTICS" GROUP BY event_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '07_returns.return_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT return_id FROM sales_raw."07_RETURNS_CANCELLATIONS" GROUP BY return_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '08_reviews.review_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT review_id FROM sales_raw."08_CUSTOMER_REVIEWS" GROUP BY review_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '09_inventory.snapshot_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT snapshot_id FROM sales_raw."09_INVENTORY_SNAPSHOTS" GROUP BY snapshot_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '10_marketing.campaign_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT campaign_id FROM sales_raw."10_MARKETING_CAMPAIGNS" GROUP BY campaign_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '11_support.ticket_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT ticket_id FROM sales_raw."11_SUPPORT_TICKETS" GROUP BY ticket_id HAVING COUNT(*) > 1
)

UNION ALL

SELECT '12_cancellations.cancellation_id' AS table_key, COUNT(*) AS duplicate_count
FROM (
    SELECT cancellation_id FROM sales_raw."12_CANCELLATIONS" GROUP BY cancellation_id HAVING COUNT(*) > 1
);
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2271963149.py:86: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,TABLE_KEY,DUPLICATE_COUNT
0,01_orders.order_id,0
1,02_order_items.order_item_id,0
2,03_products.product_id,0
3,04_customers.customer_id,0
4,05_sellers.seller_id,0
5,06_shipping.event_id,0
6,07_returns.return_id,0
7,08_reviews.review_id,0
8,09_inventory.snapshot_id,0
9,10_marketing.campaign_id,0


## 5. Referential Integrity

Check for orphan records - foreign keys that don't match any primary key in the referenced table.

In [34]:
# 5.1 Orders: Foreign key checks
query = """
-- Orders referencing non-existent customers
SELECT 
    'orders.customer_id -> customers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."01_ORDERS" o
LEFT JOIN sales_raw."04_CUSTOMERS" c ON o.customer_id = c.customer_id
WHERE o.customer_id IS NOT NULL AND c.customer_id IS NULL

UNION ALL

-- Orders referencing non-existent sellers
SELECT 
    'orders.seller_id -> sellers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."01_ORDERS" o
LEFT JOIN sales_raw."05_SELLERS" s ON o.seller_id = s.seller_id
WHERE o.seller_id IS NOT NULL AND s.seller_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\241960823.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,orders.customer_id -> customers,0
1,orders.seller_id -> sellers,0


In [35]:
# 5.2 Order Items: Foreign key checks
query = """
-- Order items referencing non-existent orders
SELECT 
    'order_items.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."02_ORDER_ITEMS" oi
LEFT JOIN sales_raw."01_ORDERS" o ON oi.order_id = o.order_id
WHERE oi.order_id IS NOT NULL AND o.order_id IS NULL

UNION ALL

-- Order items referencing non-existent products
SELECT 
    'order_items.product_id -> products' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."02_ORDER_ITEMS" oi
LEFT JOIN sales_raw."03_PRODUCTS" p ON oi.product_id = p.product_id
WHERE oi.product_id IS NOT NULL AND p.product_id IS NULL

UNION ALL

-- Order items referencing non-existent sellers
SELECT 
    'order_items.seller_id -> sellers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."02_ORDER_ITEMS" oi
LEFT JOIN sales_raw."05_SELLERS" s ON oi.seller_id = s.seller_id
WHERE oi.seller_id IS NOT NULL AND s.seller_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\3359655578.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,order_items.order_id -> orders,0
1,order_items.product_id -> products,0
2,order_items.seller_id -> sellers,0


In [36]:
# 5.3 Shipping Logistics: Foreign key checks
query = """
-- Shipping events referencing non-existent orders
SELECT 
    'shipping_logistics.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."06_SHIPPING_LOGISTICS" sl
LEFT JOIN sales_raw."01_ORDERS" o ON sl.order_id = o.order_id
WHERE sl.order_id IS NOT NULL AND o.order_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2649803549.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,shipping_logistics.order_id -> orders,0


In [37]:
# 5.4 Returns/Cancellations: Foreign key checks
query = """
-- Returns referencing non-existent orders
SELECT 
    'returns.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."07_RETURNS_CANCELLATIONS" r
LEFT JOIN sales_raw."01_ORDERS" o ON r.order_id = o.order_id
WHERE r.order_id IS NOT NULL AND o.order_id IS NULL

UNION ALL

-- Returns referencing non-existent order items
SELECT 
    'returns.order_item_id -> order_items' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."07_RETURNS_CANCELLATIONS" r
LEFT JOIN sales_raw."02_ORDER_ITEMS" oi ON r.order_item_id = oi.order_item_id
WHERE r.order_item_id IS NOT NULL AND oi.order_item_id IS NULL

UNION ALL

-- Returns referencing non-existent products
SELECT 
    'returns.product_id -> products' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."07_RETURNS_CANCELLATIONS" r
LEFT JOIN sales_raw."03_PRODUCTS" p ON r.product_id = p.product_id
WHERE r.product_id IS NOT NULL AND p.product_id IS NULL

UNION ALL

-- Returns referencing non-existent customers
SELECT 
    'returns.customer_id -> customers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."07_RETURNS_CANCELLATIONS" r
LEFT JOIN sales_raw."04_CUSTOMERS" c ON r.customer_id = c.customer_id
WHERE r.customer_id IS NOT NULL AND c.customer_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\4135245115.py:42: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,returns.order_id -> orders,0
1,returns.order_item_id -> order_items,0
2,returns.product_id -> products,0
3,returns.customer_id -> customers,0


In [38]:
# 5.5 Customer Reviews: Foreign key checks
query = """
-- Reviews referencing non-existent orders
SELECT 
    'reviews.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."08_CUSTOMER_REVIEWS" cr
LEFT JOIN sales_raw."01_ORDERS" o ON cr.order_id = o.order_id
WHERE cr.order_id IS NOT NULL AND o.order_id IS NULL

UNION ALL

-- Reviews referencing non-existent customers
SELECT 
    'reviews.customer_id -> customers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."08_CUSTOMER_REVIEWS" cr
LEFT JOIN sales_raw."04_CUSTOMERS" c ON cr.customer_id = c.customer_id
WHERE cr.customer_id IS NOT NULL AND c.customer_id IS NULL

UNION ALL

-- Reviews referencing non-existent products
SELECT 
    'reviews.product_id -> products' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."08_CUSTOMER_REVIEWS" cr
LEFT JOIN sales_raw."03_PRODUCTS" p ON cr.product_id = p.product_id
WHERE cr.product_id IS NOT NULL AND p.product_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\3762265437.py:32: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,reviews.order_id -> orders,0
1,reviews.customer_id -> customers,0
2,reviews.product_id -> products,0


In [39]:
# 5.6 Inventory Snapshots: Foreign key checks
query = """
-- Inventory snapshots referencing non-existent products
SELECT 
    'inventory.product_id -> products' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."09_INVENTORY_SNAPSHOTS" inv
LEFT JOIN sales_raw."03_PRODUCTS" p ON inv.product_id = p.product_id
WHERE inv.product_id IS NOT NULL AND p.product_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\795997687.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,inventory.product_id -> products,0


In [40]:
# 5.7 Support Tickets: Foreign key checks
query = """
-- Support tickets referencing non-existent orders
SELECT 
    'support_tickets.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."11_SUPPORT_TICKETS" st
LEFT JOIN sales_raw."01_ORDERS" o ON st.order_id = o.order_id
WHERE st.order_id IS NOT NULL AND o.order_id IS NULL

UNION ALL

-- Support tickets referencing non-existent customers
SELECT 
    'support_tickets.customer_id -> customers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."11_SUPPORT_TICKETS" st
LEFT JOIN sales_raw."04_CUSTOMERS" c ON st.customer_id = c.customer_id
WHERE st.customer_id IS NOT NULL AND c.customer_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\2741726177.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,support_tickets.order_id -> orders,0
1,support_tickets.customer_id -> customers,0


In [41]:
# 5.8 Cancellations: Foreign key checks
query = """
-- Cancellations referencing non-existent orders
SELECT 
    'cancellations.order_id -> orders' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."12_CANCELLATIONS" ca
LEFT JOIN sales_raw."01_ORDERS" o ON ca.order_id = o.order_id
WHERE ca.order_id IS NOT NULL AND o.order_id IS NULL

UNION ALL

-- Cancellations referencing non-existent customers
SELECT 
    'cancellations.customer_id -> customers' AS relationship,
    COUNT(*) AS orphan_count
FROM sales_raw."12_CANCELLATIONS" ca
LEFT JOIN sales_raw."04_CUSTOMERS" c ON ca.customer_id = c.customer_id
WHERE ca.customer_id IS NOT NULL AND c.customer_id IS NULL;
"""

df = pd.read_sql(query, conn)
display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\659414762.py:22: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,RELATIONSHIP,ORPHAN_COUNT
0,cancellations.order_id -> orders,0
1,cancellations.customer_id -> customers,0


## 6. Summary: All Validity Checks

Consolidated view of all data quality issues found.

In [42]:
# 6.1 Summary of all validity issues (only showing non-zero violations)
query = """
WITH validity_checks AS (
    -- Impossible dates
    SELECT 'Date: delivered_before_ordered' AS check_name, COUNT(*) AS violations
    FROM sales_raw."01_ORDERS" WHERE delivered_date < order_date
    UNION ALL
    SELECT 'Date: approved_before_ordered', COUNT(*) FROM sales_raw."01_ORDERS" WHERE approval_date < order_date
    UNION ALL
    SELECT 'Date: shipped_before_approved', COUNT(*) FROM sales_raw."01_ORDERS" WHERE shipped_date < approval_date
    UNION ALL
    SELECT 'Date: delivered_before_shipped', COUNT(*) FROM sales_raw."01_ORDERS" WHERE delivered_date < shipped_date
    UNION ALL
    
    -- Status consistency
    SELECT 'Status: delivered_no_date', COUNT(*) FROM sales_raw."01_ORDERS" WHERE order_status = 'Delivered' AND delivered_date IS NULL
    UNION ALL
    SELECT 'Status: has_date_not_delivered', COUNT(*) FROM sales_raw."01_ORDERS" WHERE delivered_date IS NOT NULL AND order_status != 'Delivered'
    UNION ALL
    
    -- Invalid values
    SELECT 'Value: negative_unit_price', COUNT(*) FROM sales_raw."02_ORDER_ITEMS" WHERE unit_price <= 0
    UNION ALL
    SELECT 'Value: negative_order_total', COUNT(*) FROM sales_raw."01_ORDERS" WHERE order_total <= 0
    UNION ALL
    SELECT 'Value: cost_exceeds_price', COUNT(*) FROM sales_raw."03_PRODUCTS" WHERE cost_price > price
    UNION ALL
    
    -- Referential integrity (key relationships)
    SELECT 'FK: orders->customers orphans', COUNT(*)
    FROM sales_raw."01_ORDERS" o LEFT JOIN sales_raw."04_CUSTOMERS" c ON o.customer_id = c.customer_id
    WHERE o.customer_id IS NOT NULL AND c.customer_id IS NULL
    UNION ALL
    SELECT 'FK: order_items->orders orphans', COUNT(*)
    FROM sales_raw."02_ORDER_ITEMS" oi LEFT JOIN sales_raw."01_ORDERS" o ON oi.order_id = o.order_id
    WHERE oi.order_id IS NOT NULL AND o.order_id IS NULL
    UNION ALL
    SELECT 'FK: order_items->products orphans', COUNT(*)
    FROM sales_raw."02_ORDER_ITEMS" oi LEFT JOIN sales_raw."03_PRODUCTS" p ON oi.product_id = p.product_id
    WHERE oi.product_id IS NOT NULL AND p.product_id IS NULL
)
SELECT check_name, violations
FROM validity_checks
WHERE violations > 0
ORDER BY violations DESC;
"""

df = pd.read_sql(query, conn)
if df.empty:
    print("No validity issues found!")
else:
    display(df)

C:\Users\frase\AppData\Local\Temp\ipykernel_48632\1243559883.py:48: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)


,CHECK_NAME,VIOLATIONS
0,Date: delivered_before_ordered,130
1,Date: delivered_before_shipped,130


In [43]:
conn.close()